## Model Selection: Gemma4 Efficient 2B

### Preparation

In [1]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.4 MB/s eta 0:00:00


In [2]:
!pip install -q -U transformers peft trl datasets scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.4 MB/s eta 0:00:00


### Execution

In [3]:
import os
import sys
import gc
import time
import torch
import random
import numpy as np
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from sklearn.metrics import f1_score, accuracy_score, classification_report
from huggingface_hub import login

# ---------------------------------------------------------
# 1. HARDWARE CHECK & MEMORY CLEANSING
# ---------------------------------------------------------
if not torch.cuda.is_available():
    raise SystemError("NO GPU DETECTED! Select T4 GPU in Runtime settings.")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print("GPU memory cleared. Seeds locked.")

# ---------------------------------------------------------
# 2. ENVIRONMENT & AUTHENTICATION
# ---------------------------------------------------------
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive', force_remount=True)

    REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"
    hf_cache_dir = "/content/drive/MyDrive/progettoLLM/hf_cache"
    os.makedirs(hf_cache_dir, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache_dir
    os.chdir(REPO_DIR)

    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Environment and Hugging Face Auth Configured.")
except Exception as e:
    print(f"Setup warning: {e}")

# ---------------------------------------------------------
# 3. DATASET FORMATTING (GEMMA 4 'THINK' MODE)
# ---------------------------------------------------------
VALID_LABELS = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
label2id = {label: i for i, label in enumerate(VALID_LABELS)}
MAJORITY_CLASS_ID = label2id["Ambivalent"]

model_id = "google/gemma-4-e2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
# Gemma uses <eos> as the pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(example, is_inference=False):
    # Gemma 4 responds incredibly well to ChatML formatting
    chat = [
        {"role": "system", "content": "You are an expert political analyst. Classify the clarity of the interview answer. You must choose EXACTLY ONE of the following labels: [Clear Reply, Ambivalent, Clear Non-Reply]."},
        {"role": "user", "content": f"Question: {example['question']}\nAnswer: {example['interview_answer']}"}
    ]

    # Apply the native Gemma template
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

    if not is_inference:
        # Trigger the native thinking block during training
        prompt += f"<|think|>\nAnalyzing the response against the question...\n</|think|>\n{example['clarity_label']}{tokenizer.eos_token}"
    else:
        # Just prompt the thinking start tag for inference
        prompt += "<|think|>\n"

    return prompt

print("Loading and formatting datasets...")
train_dataset = load_from_disk("dataset/QEvasion/train")
test_dataset = load_from_disk("dataset/QEvasion/test")

train_dataset = train_dataset.map(lambda x: {"text": format_prompt(x, is_inference=False)})
test_dataset = test_dataset.map(lambda x: {"text": format_prompt(x, is_inference=False)})

# ---------------------------------------------------------
# 4. MODEL SETUP (NATIVE FP16, NO QLORA NEEDED)
# ---------------------------------------------------------
print("Loading Gemma 4 E2B in Native FP16...")
# Because E2B is so small, we can load it entirely in FP16 on the T4
# without BitsAndBytes quantization, preventing scaling crashes.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": 0},
    dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

# We still use standard LoRA to train it faster
# We still use standard LoRA to train it faster
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    # THE FIX: Reach inside the Gemma4 wrapper to target the standard PyTorch Linear layer
    target_modules=[
        "q_proj.linear",
        "k_proj.linear",
        "v_proj.linear",
        "o_proj.linear",
        "gate_proj.linear",
        "up_proj.linear",
        "down_proj.linear"
    ],
    task_type=TaskType.CAUSAL_LM,
)
peft_model = get_peft_model(model, lora_config)

# ---------------------------------------------------------
# 5. TRAINING VIA SFTTrainer
# ---------------------------------------------------------
training_args = SFTConfig(
    output_dir="./gemma_clarity_results",
    dataset_text_field="text",
    max_length=512,
    eval_strategy="no",
    logging_steps=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,  # 1 Epoch for fast benchmarking
    warmup_steps=50,
    weight_decay=0.01,

    fp16=True,  # Safe, native float16 math for T4
    bf16=False,

    gradient_checkpointing=True,
    dataloader_num_workers=2,
    save_strategy="epoch",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    args=training_args,
)

print("\nStarting Gemma 4 SFT Training...")
start_time = time.time()

trainer.train()

end_time = time.time()
training_duration_minutes = (end_time - start_time) / 60
print(f"Training completed in {training_duration_minutes:.2f} minutes.")

# ---------------------------------------------------------
# 6. INFERENCE EVALUATION
# ---------------------------------------------------------
print("\nStarting Inference Evaluation...")

def extract_label(generated_text):
    # Strip the <|think|> block out before searching for the label
    clean_text = generated_text.split("</|think|>")[-1].lower() if "</|think|>" in generated_text else generated_text.lower()

    found_labels = []
    for label in VALID_LABELS:
        if label.lower() in clean_text:
            found_labels.append(label)

    if len(found_labels) >= 1:
        return label2id[found_labels[-1]]
    else:
        return MAJORITY_CLASS_ID

peft_model.eval()
true_labels = []
pred_labels = []

for item in test_dataset:
    inference_prompt = format_prompt(item, is_inference=True)
    inputs = tokenizer(inference_prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")

    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs,
            max_new_tokens=150, # Give it room to think
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    pred_id = extract_label(generated_text)
    true_id = label2id[item['clarity_label']]

    pred_labels.append(pred_id)
    true_labels.append(true_id)

# ---------------------------------------------------------
# 7. FINAL METRICS & EFFICIENCY REPORT
# ---------------------------------------------------------
accuracy = accuracy_score(true_labels, pred_labels)
macro_f1 = f1_score(true_labels, pred_labels, average='macro')
weighted_f1 = f1_score(true_labels, pred_labels, average='weighted')

peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\n==============================================")
print("--- FINAL GEMMA 4 E2B RESULTS ---")
print("==============================================")
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")
print("----------------------------------------------")
print("--- EFFICIENCY METRICS ---")
print(f"Training Time:   {training_duration_minutes:.2f} minutes")
print(f"Peak VRAM Usage: {peak_memory_gb:.2f} GB")
print("==============================================\n")

print("Detailed Classification Report:\n", classification_report(true_labels, pred_labels, target_names=VALID_LABELS))

GPU memory cleared. Seeds locked.
Mounted at /content/drive
Environment and Hugging Face Auth Configured.


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Adding EOS to train dataset:   0%|          | 0/3448 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3448 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.



Starting Gemma 4 SFT Training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 1022.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 577.81 MiB is free. Including non-PyTorch memory, this process has 14.00 GiB memory in use. Of the allocated memory 13.21 GiB is allocated by PyTorch, and 664.50 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)